# Register the v1 NWM point datasets and copy their legacy history

Sets `public` up for the `v1-ingest-nwm-*` flows (`prefect-workflows/v1/flows/ingest_nwm.py`):

| v1 dataset (`public`) | what |
| --- | --- |
| `nwm31_<configuration>` (14) | one per legacy configuration, legacy's names: forecasts (`reference_time`; medium range mem1 also `member`) and analysis (`simulated`, no `reference_time`) |
| `nwm31_short_range_series` | the series table they all share (`nwm31-<feature_id>` keys) |
| `usgs_nwm31` (crosswalk) | the gages to their nwm31 reaches (legacy's crosswalk): pairs the gages with every dataset above |

Then copies the legacy history of every one of them (`COPY`; the nwmd notebook and dashboard use
them all) into its dataset, a month at a time, so the flows carry on from it.

**Run it as a platform admin**, on KinD first (its seed already has `nwm31_short_range`, the
series table and `usgs_nwm31`: they're left as they are), then use1.

**Spark writes the rows straight into the `public` Iceberg tables** (as `01_copy_configuration_to_public`):
the API takes at most 64 MB per write. A one-time admin migration, not a pattern (ADR-0004).

**Re-runnable:** registering skips datasets that exist; the crosswalk's links are written only
while it's empty; series are added, never removed; each month copied replaces that month.

**After it:** run each `v1-ingest-nwm-*` by hand once with `start` at the copy's last cycle (or
`lookback_days` covering the gap), then turn its schedule on. Legacy keeps ingesting until the
comparison matches.

Start with **2. Count**: it writes nothing.

In [ ]:
import os

import pandas as pd
import requests
import teehr
from pyspark.sql import functions as F
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
VERSION = "nwm31"
#: GCS configuration directory -> v1 dataset (legacy's configuration name: no `_memN`).
CONFIGURATIONS = {
    "short_range": "short_range", "short_range_alaska": "short_range_alaska",
    "short_range_hawaii": "short_range_hawaii", "short_range_puertorico": "short_range_puertorico",
    "medium_range_mem1": "medium_range", "medium_range_blend": "medium_range_blend",
    "medium_range_alaska_mem1": "medium_range_alaska",
    "medium_range_blend_alaska": "medium_range_blend_alaska",
    "analysis_assim_no_da": "analysis_assim_no_da",
    "analysis_assim_alaska_no_da": "analysis_assim_alaska_no_da",
    "analysis_assim_extend_no_da": "analysis_assim_extend_no_da",
    "analysis_assim_extend_alaska_no_da": "analysis_assim_extend_alaska_no_da",
    "analysis_assim_hawaii_no_da": "analysis_assim_hawaii_no_da",
    "analysis_assim_puertorico_no_da": "analysis_assim_puertorico_no_da",
}
DATASETS = {gcs: f"{VERSION}_{name}" for gcs, name in CONFIGURATIONS.items()}
# The first registered makes the series table (`<name>_series`); the rest share it.
ANCHOR = f"{VERSION}_short_range"
SERIES = f"{ANCHOR}_series"
CROSSWALK, OBSERVED = f"usgs_{VERSION}", "usgs_streamflow"

# Datasets whose legacy history to copy (their legacy configuration has the same name): all of
# them, by default. Narrow it to re-copy some.
COPY = list(DATASETS.values())

# The control plane API, from inside the cluster (KinD and use1 alike). From outside:
# https://control-plane.teehr.rtiamanzi.org (use1), https://control-plane.teehr.local.app.garden (KinD).
CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed

LEGACY, PUBLIC = "iceberg.teehr", "iceberg.public"
SOURCE = ("NOAA National Water Model operational channel_rt (GCS national-water-model); "
          "history before the v1 ingests copied from the legacy TEEHR warehouse")

In [ ]:
# A cluster for the copy (14 configurations since nwm31 began); local mode is enough to count
# and register (comment the arguments out).
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=16,
    executor_memory="16g",
    executor_cores=4,
)
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")  # months are UTC, as Iceberg's partitions


def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def exists(name):
    return call("GET", f"/projects/public/datasets/{name}").status_code == 200


def register(body):
    if exists(body["name"]):
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


def month_range(column, month):
    """A time range filter for one UTC month (prunes the scan, unlike date_format)."""
    start = pd.Period(month, freq="M")
    return (f"{column} >= TIMESTAMP '{start.start_time:%Y-%m-%d} 00:00:00' "
            f"AND {column} < TIMESTAMP '{(start + 1).start_time:%Y-%m-%d} 00:00:00'")


me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")

## 2. Count (writes nothing)

What the legacy warehouse holds for each configuration: rows per month (of `reference_time`
for forecasts, `value_time` for analysis), with units and members from the latest month.

In [ ]:
%%time
from datetime import date, datetime, timedelta


def is_analysis(dataset):
    return "analysis_assim" in dataset


def time_column(dataset):
    return "value_time" if is_analysis(dataset) else "reference_time"


def month_of(field, value):
    """The month (YYYY-MM) of a partition value of an Iceberg time transform."""
    if value is None:
        return None
    if isinstance(value, (date, datetime)):
        return f"{value:%Y-%m}"
    if field.endswith("_month"):  # months since 1970-01
        return f"{1970 + value // 12:04d}-{value % 12 + 1:02d}"
    if field.endswith("_day"):
        return f"{date(1970, 1, 1) + timedelta(days=value):%Y-%m}"
    if field.endswith("_hour"):
        return f"{datetime(1970, 1, 1) + timedelta(hours=value):%Y-%m}"
    return None


# Rows per month of value_time, from Iceberg's partition metadata (no scan) when legacy is
# partitioned by configuration and time; otherwise by scanning.
parts = spark.table(f"{LEGACY}.secondary_timeseries.partitions")
fields = parts.schema["partition"].dataType.fieldNames()
config = next((f for f in fields if f.startswith("configuration_name")), None)
time = next((f for f in fields if f.startswith("value_time")), None)
names = ", ".join(f"'{d}'" for d in DATASETS.values())
if config and time and not time.endswith("_year"):
    legacy = (parts.where(F.col(f"partition.{config}").isin(list(DATASETS.values())))
              .select(F.col(f"partition.{config}").alias("dataset"),
                      F.col(f"partition.{time}").alias("t"), F.col("record_count").alias("rows"))
              .toPandas())
    legacy["month"] = [month_of(time, t) for t in legacy.t]
else:
    print(f"legacy partitioned by {fields}: scanning")
    legacy = spark.sql(f"""
        SELECT configuration_name AS dataset, date_format(value_time, 'yyyy-MM') AS month,
               count(*) AS rows
        FROM {LEGACY}.secondary_timeseries WHERE configuration_name IN ({names}) GROUP BY 1, 2
    """).toPandas()
summary = legacy.groupby("dataset").agg(first=("month", "min"), last=("month", "max"),
                                        rows=("rows", "sum"))
summary["copy"] = summary.index.isin(COPY)
missing = [d for d in COPY if d not in summary.index]
if missing:
    print(f"not in legacy, not copied: {missing}")
    COPY = [d for d in COPY if d in summary.index]
summary

In [ ]:
# Units, members and reference times of each dataset to copy (its latest month).
for dataset in COPY:
    last = summary.loc[dataset, "last"]
    spark.sql(f"""
        SELECT configuration_name, unit_name, variable_name, member,
               count(*) AS rows, count(reference_time) AS with_reference_time
        FROM {LEGACY}.secondary_timeseries
        WHERE configuration_name = '{dataset}' AND {month_range("value_time", last)}
        GROUP BY 1, 2, 3, 4 ORDER BY 1, 2, 3, 4
    """).show(truncate=False)

## 3. Register the datasets and the crosswalk (as you)

Streamflow in m3/s (legacy's `m^3/s`: the counts above must show only that unit). The anchor
first: it makes the series table the others share. Medium range mem1 configurations have a
`member` column (`"1"`), as legacy stored them.

In [ ]:
def body(gcs, dataset):
    analysis = is_analysis(dataset)
    member = "_mem" in gcs
    return {
        "type": "timeseries", "name": dataset,
        "role": "simulated" if analysis else "forecast",
        "variable": "streamflow", "statistic": "inst",
        "optional_columns": ["member"] if member else [],
        **({} if dataset == ANCHOR else {"series_table": SERIES}),
        "source": SOURCE,
        "description": (f"NWM {VERSION[3]}.{VERSION[4:]} {gcs.replace('_', ' ')} streamflow at the "
                        "gages' reaches" + (": the latest cycle's value per time" if analysis else "")),
    }


anchor = next(g for g, d in DATASETS.items() if d == ANCHOR)
register(body(anchor, ANCHOR))
for gcs, dataset in DATASETS.items():
    if dataset != ANCHOR:
        register(body(gcs, dataset))

register({"type": "crosswalk", "name": CROSSWALK, "left": OBSERVED, "right": ANCHOR,
          "source": "Legacy TEEHR warehouse (location crosswalks)",
          "description": f"USGS gages to their {VERSION} reaches"})

## 4. The crosswalk's links and the series

The links from legacy's crosswalk, only while the crosswalk is empty (KinD's seed has its
own). Every linked reach is then added to the series table (missing ones only), so the series
are there before any rows: the ingests add any others they write.

In [ ]:
%%time
if spark.table(f"{PUBLIC}.{CROSSWALK}").limit(1).count() == 0:
    spark.sql(f"""
        SELECT DISTINCT primary_location_id AS left_key, secondary_location_id AS right_key
        FROM {LEGACY}.location_crosswalks
        WHERE primary_location_id LIKE 'usgs-%' AND secondary_location_id LIKE '{VERSION}-%'
    """).writeTo(f"{PUBLIC}.{CROSSWALK}").append()
print(f"{CROSSWALK}: {spark.table(f'{PUBLIC}.{CROSSWALK}').count():,} links")

spark.sql(f"""
    MERGE INTO {PUBLIC}.{SERIES} t
    USING (SELECT DISTINCT right_key AS series_key FROM {PUBLIC}.{CROSSWALK}) s
    ON t.series_key = s.series_key
    WHEN NOT MATCHED THEN INSERT (series_key, location_key, location_layer, period, resolution)
        VALUES (s.series_key, NULL, NULL, NULL, NULL)
""")
print(f"{SERIES}: {spark.table(f'{PUBLIC}.{SERIES}').count():,} series")

## 5. Copy the history

A month at a time, each replacing that month of the v1 table (its partitions: `reference_time`
for forecasts, `value_time` for analysis). Values are rounded to 0.01 m3/s, the files'
precision: legacy stored float32 (`12.34` as `12.3400001525...`), the v1 ingests store the exact
`12.34`. Analysis has no `reference_time`, as in the v1 ingests (legacy's is already null).

In [ ]:
MAX_LEAD = "16D"  # longer than any configuration's longest lead (medium range: 10 days)
START = None      # e.g. "2026-10" to resume from a month


def columns(dataset):
    member = "member" in [f.name for f in spark.table(f"{PUBLIC}.{dataset}").schema]
    return ["location_id AS series_key",
            *([] if is_analysis(dataset) else ["reference_time"]),
            "value_time", *(["member"] if member else []),
            "ROUND(CAST(value AS DOUBLE), 2) AS value"]


def copy_month(dataset, month):
    """Replace one UTC month of the v1 dataset with the legacy rows."""
    where = month_range(time_column(dataset), month)
    if not is_analysis(dataset):
        # Legacy is partitioned by month of value_time: bound it too, so the scan reads two
        # months of partitions, not all.
        start = pd.Period(month, freq="M").start_time
        end = (pd.Period(month, freq="M") + 1).start_time + pd.Timedelta(MAX_LEAD)
        where += (f" AND value_time >= TIMESTAMP '{start:%Y-%m-%d} 00:00:00'"
                  f" AND value_time < TIMESTAMP '{end:%Y-%m-%d %H:%M:%S}'")
    rows = spark.sql(f"""
        SELECT {", ".join(columns(dataset))} FROM {LEGACY}.secondary_timeseries
        WHERE configuration_name = '{dataset}' AND {where} AND value IS NOT NULL
    """)
    rows.writeTo(f"{PUBLIC}.{dataset}").overwritePartitions()

In [ ]:
%%time
for dataset in COPY:
    first, last = summary.loc[dataset, ["first", "last"]]
    for month in (str(m) for m in pd.period_range(first, last, freq="M")):
        if START and month < START:
            continue
        copy_month(dataset, month)
        print(f"{dataset} {month}: copied")

## 6. Check: v1 against legacy

Rows per month of each copied dataset against legacy's (every non-null row is copied), and
the last cycle copied: where the ingests take over.

In [ ]:
%%time
def per_month(table, column, name, where="TRUE"):
    return spark.sql(f"""
        SELECT date_format({column}, 'yyyy-MM') AS month, count(*) AS {name}
        FROM {table} WHERE {where} AND value IS NOT NULL GROUP BY 1
    """).toPandas()


for dataset in COPY:
    column = time_column(dataset)
    both = per_month(f"{LEGACY}.secondary_timeseries", column, "legacy",
                     f"configuration_name = '{dataset}'").merge(
        per_month(f"{PUBLIC}.{dataset}", column, "v1"), on="month", how="outer").fillna(0)
    both["matches"] = both.legacy == both.v1
    last = spark.sql(f"SELECT max({column}) AS t FROM {PUBLIC}.{dataset}").first().t
    print(f"{dataset}: {int(both.v1.sum()):,} rows; every month matches: {both.matches.all()}; "
          f"last {column}: {last}")
    display(both[~both.matches])